# Üye 3 — Gradio Arayüz
YZM358 Doğal Dil İşleme Projesi

**Çalıştırma sırası:**
1. Kurulum cell'ini çalıştır → kernel otomatik yeniden başlar
2. Pipeline cell'ini çalıştır
3. Gradio arayüz cell'ini çalıştır


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q node2vec==0.4.6
!pip install -q numpy==1.26.4 --force-reinstall
!pip install -q tree-sitter==0.21.3 tree-sitter-python==0.21.0 tree-sitter-java==0.21.0
!pip install -q transformers==4.41.0 sentencepiece networkx gradio
!pip install -q faiss-cpu

import os
os.kill(os.getpid(), 9)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
mapclassify 2.10.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0

In [ ]:
import sys, re, pickle
import numpy as np
import networkx as nx
import torch
import faiss
import tree_sitter_python as tspython
import tree_sitter_java as tsjava
from tree_sitter import Language, Parser
from node2vec import Node2Vec
from transformers import RobertaTokenizer, T5ForConditionalGeneration

DRIVE = "/content/drive/MyDrive/CodeReviewBot"
PY_LANGUAGE   = Language(tspython.language(), "python")
JAVA_LANGUAGE = Language(tsjava.language(), "java")
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = RobertaTokenizer.from_pretrained(f"{DRIVE}/model_ciktisi/final_model")
model     = T5ForConditionalGeneration.from_pretrained(f"{DRIVE}/model_ciktisi/final_model").to(device)
model.eval()
print("[Pipeline] Review model yuklendi!")

faiss_index = faiss.read_index(f"{DRIVE}/model_ciktisi/faiss_index.bin")
with open(f"{DRIVE}/model_ciktisi/corpus_data.pkl", "rb") as f:
    corpus_data = pickle.load(f)
print(f"[Pipeline] FAISS yuklendi! ({faiss_index.ntotal} vektor)")

repair_tokenizer = RobertaTokenizer.from_pretrained(f"{DRIVE}/model_ciktisi/repair_model/")
repair_model     = T5ForConditionalGeneration.from_pretrained(f"{DRIVE}/model_ciktisi/repair_model/").to(device)
repair_model.eval()
print("[Pipeline] Repair model yuklendi!")


def get_ast(code, lang):
    parser = Parser()
    if lang == "python":
        parser.set_language(PY_LANGUAGE)
    elif lang == "java":
        parser.set_language(JAVA_LANGUAGE)
    else:
        raise ValueError(f"Desteklenmeyen dil: {lang}")
    return parser.parse(bytes(code, "utf-8")).root_node


def ast_to_graph(node, graph=None, parent_id=None, node_id=None):
    if node_id is None:
        node_id = [0]
    if graph is None:
        graph = nx.DiGraph()
        node_id[0] = 0
    current_id = node_id[0]
    node_id[0] += 1
    graph.add_node(current_id, type=node.type,
                   text=node.text.decode("utf-8") if node.text else "")
    if parent_id is not None:
        graph.add_edge(parent_id, current_id)
    for child in node.children:
        ast_to_graph(child, graph, current_id, node_id)
    return graph


_encoding_cache = {}

def graph_to_encoding(graph, dimensions=64):
    if graph.number_of_nodes() == 0:
        return np.zeros(dimensions)
    cache_key = str(sorted(graph.edges()))
    if cache_key in _encoding_cache:
        return _encoding_cache[cache_key]
    n2v = Node2Vec(graph, dimensions=dimensions, walk_length=10,
                   num_walks=20, workers=1, quiet=True)
    m = n2v.fit(window=5, min_count=1)
    result = np.mean([m.wv[str(n)] for n in graph.nodes()], axis=0)
    _encoding_cache[cache_key] = result
    return result


def rag_ara(query_code, k=2):
    enc_in = tokenizer(query_code, return_tensors="pt",
                       truncation=True, max_length=512).to(device)
    with torch.no_grad():
        emb = model.encoder(**enc_in).last_hidden_state.mean(dim=1)
        emb = emb.cpu().numpy().astype("float32")
    _, idxs = faiss_index.search(emb, k)
    return [corpus_data[i] for i in idxs[0] if i != -1]


def kategori_belirle(review):
    r = review.lower()
    if any(k in r for k in ["security","injection","vulnerability","unsafe","attack","password","auth"]):
        return "Guvenlik"
    elif any(k in r for k in ["performance","slow","inefficient","loop","complexity","optimize","memory"]):
        return "Performans"
    elif any(k in r for k in ["naming","readability","style","format","comment","docstring","variable"]):
        return "Okunabilirlik"
    return "Genel"


def code_review(code, lang="python", verbose=False):
    try:
        ast  = get_ast(code, lang)
        graf = ast_to_graph(ast)
        encoding = graph_to_encoding(graf)
        node_tipleri = list(set(nx.get_node_attributes(graf, "type").values()))[:5]
        graf_bilgisi = (
            f"[GRAPH ANALYSIS] Nodes: {graf.number_of_nodes()}, "
            f"Edges: {graf.number_of_edges()}, "
            f"Node types: {', '.join(node_tipleri)}"
        )
    except Exception as e:
        print(f"[Uyari] Graf analizi basarisiz: {e}")
        graf = nx.DiGraph()
        graf_bilgisi = "[GRAPH ANALYSIS] Graf olusturulamadi."

    rag_ornekler = rag_ara(code, k=2)
    rag_satirlar = ["- " + str(o.get("msg", ""))[:80] for o in rag_ornekler]
    if rag_satirlar:
        rag_metni = "Similar reviews:" + chr(10) + chr(10).join(rag_satirlar)
    else:
        rag_metni = ""

    if verbose:
        print(chr(10) + "RAG Ornekleri:")
        for i, o in enumerate(rag_ornekler):
            print(f"  [{i+1}] {str(o.get('msg', ''))[:100]}")

    yarim = len(code) // 2
    prompt = chr(10).join([
        "Review this code change:",
        graf_bilgisi,
        rag_metni,
        "[OLD]: " + code[:yarim],
        "[NEW]: " + code[:400],
    ])

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_length=128, min_length=5,
            num_beams=4, no_repeat_ngram_size=3, early_stopping=True
        )
    review = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {
        "review"      : review,
        "kategori"    : kategori_belirle(review),
        "graph_nodes" : graf.number_of_nodes(),
        "graph_edges" : graf.number_of_edges(),
        "rag_ornekler": [str(o.get("msg", "")) for o in rag_ornekler]
    }


def code_repair_model_fn(code, lang="python"):
    prompt = "fix " + lang + ": " + code[:400]
    inputs = repair_tokenizer(prompt, return_tensors="pt",
                              max_length=256, truncation=True).to(device)
    with torch.no_grad():
        outputs = repair_model.generate(**inputs, max_length=256,
                                        num_beams=4, early_stopping=True)
    return repair_tokenizer.decode(outputs[0], skip_special_tokens=True)


def code_repair_kural(code, lang="python"):
    duzeltmeler = []
    yeni_kod = code
    cift_tirnak_plus = chr(34) + " +"
    tek_tirnak_plus  = chr(39) + " +"
    if "execute(" in code and (cift_tirnak_plus in code or tek_tirnak_plus in code):
        yeni_kod = re.sub(
            r'\.execute\((["\'])(.+?)\1 \+ (\w+)\)',
            r'.execute("\2?", (\3,))',
            yeni_kod
        )
        duzeltmeler.append("SQL injection duzeltildi")
    if "open(" in code and "close()" not in code and "with open" not in code:
        satirlar = yeni_kod.split(chr(10))
        yeni_satirlar = []
        for satir in satirlar:
            if satir.strip().startswith("return") and "dosya" not in satir:
                yeni_satirlar.append("    dosya.close()")
            yeni_satirlar.append(satir)
        yeni_kod = chr(10).join(yeni_satirlar)
        duzeltmeler.append("Dosya kapatma eklendi")
    s1234d = chr(34) + "1234" + chr(34)
    s1234t = chr(39) + "1234" + chr(39)
    sadmind = chr(34) + "admin" + chr(34)
    sadmint = chr(39) + "admin" + chr(39)
    if any(k in code for k in [s1234d, s1234t, sadmind, sadmint]):
        yeni_kod = "# Sabit sifre! Environment variable kullanin." + chr(10) + yeni_kod
        duzeltmeler.append("Sabit sifre tespit edildi")
    if not duzeltmeler:
        return code, ["Bilinen hata kalibi bulunamadi"]
    return yeni_kod, duzeltmeler


def code_repair(code, lang="python"):
    if lang == "java":
        return code_repair_model_fn(code, lang), ["Java repair modeli kullanildi"]
    yeni_kod, mesajlar = code_repair_kural(code, lang)
    if "Bilinen hata kalibi bulunamadi" in mesajlar:
        ai_duzeltme = code_repair_model_fn(code, lang)
        return ai_duzeltme, ["Yapay Zeka (CodeT5+) ile duzeltildi"]
    return yeni_kod, mesajlar


print("Pipeline hazir!")


[Pipeline] Review model yuklendi!
[Pipeline] FAISS yuklendi! (2000 vektor)
[Pipeline] Repair model yuklendi!
Pipeline hazir!


In [ ]:
import gradio as gr
import json

with open(f"{DRIVE}/model_ciktisi/metrikler.json") as mf:
    metrikler = json.load(mf)

bleu      = metrikler.get("bleu", 0)
bertscore = metrikler.get("bertscore_f1", 0)
f1        = metrikler.get("f1", 0)

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');
* { font-family: Inter, sans-serif !important; box-sizing: border-box; }
body, .gradio-container { background: #0a0a14 !important; margin: 0 !important; padding: 0 !important; max-width: 100% !important; }
footer { display: none !important; }
body.light-mode, body.light-mode .gradio-container { background: #f8fafc !important; }
body.light-mode textarea { background: #ffffff !important; color: #0f172a !important; }
.analiz-btn button { background: linear-gradient(135deg,#7C3AED,#4F46E5) !important; color:white !important; border:none !important; border-radius:10px !important; font-size:15px !important; font-weight:600 !important; padding:14px !important; width:100% !important; }
.send-btn button { background:#7C3AED !important; color:white !important; border:none !important; border-radius:10px !important; font-weight:600 !important; padding:10px 18px !important; }
textarea { background:#0a0a14 !important; color:#e2e8f0 !important; border:1px solid #1e1e2e !important; border-radius:10px !important; font-family: JetBrains Mono, monospace !important; font-size:13px !important; }
.metric-bar-bg { height:5px; background:#1e1e2e; border-radius:3px; overflow:hidden; margin-top:4px; }
.metric-bar-fill { height:100%; border-radius:3px; }
"""


def metrik_html_olustur():
    f1_pct   = round(f1 * 100)
    bert_pct = round(bertscore * 100)
    bleu_pct = min(round(bleu * 100), 100)
    return (
        "<div style='display:flex;align-items:center;gap:16px;'>"
        "<div style='flex:1;min-width:0;'>"
        f"<div style='display:flex;justify-content:space-between;color:#94a3b8;font-size:12px;margin-bottom:4px;'>BERTScore <span style='color:#7C3AED;font-weight:600;'>{bertscore:.2f}</span></div>"
        f"<div class='metric-bar-bg'><div class='metric-bar-fill' style='width:{bert_pct}%;background:linear-gradient(90deg,#7C3AED,#4F46E5);'></div></div>"
        f"<div style='display:flex;justify-content:space-between;color:#94a3b8;font-size:12px;margin:8px 0 4px 0;'>BLEU <span style='color:#06b6d4;font-weight:600;'>{bleu:.2f}</span></div>"
        f"<div class='metric-bar-bg'><div class='metric-bar-fill' style='width:{bleu_pct}%;background:linear-gradient(90deg,#06b6d4,#3b82f6);'></div></div>"
        f"<div style='display:flex;justify-content:space-between;color:#94a3b8;font-size:12px;margin:8px 0 4px 0;'>F1 Score <span style='color:#22c55e;font-weight:600;'>{f1:.2f}</span></div>"
        f"<div class='metric-bar-bg'><div class='metric-bar-fill' style='width:{f1_pct}%;background:linear-gradient(90deg,#22c55e,#06b6d4);'></div></div>"
        "</div></div>"
    )


def dil_tespit(kod):
    if any(k in kod for k in ["public class", "System.out", "void main", "public static"]):
        return "java"
    return "python"


def analiz_et(kod, gecmis):
    if not kod.strip():
        return gecmis, gecmis, metrik_html_olustur()
    dil = dil_tespit(kod)
    try:
        sonuc = code_review(kod, dil)
        parcalar = [
            f"**Analysis ({dil.upper()})**",
            sonuc["review"],
            "",
            "**Kod Istatistikleri**",
            f"- Graf dugum sayisi: {sonuc['graph_nodes']}",
            f"- Graf kenar sayisi: {sonuc['graph_edges']}",
            f"- Kategori: {sonuc['kategori']}",
            "",
            "**Benzer Gecmis Incelemeler (RAG)**",
        ] + ["- " + r[:80] for r in sonuc["rag_ornekler"]]
        cevap = chr(10).join(parcalar)
    except Exception as e:
        cevap = "Hata: " + str(e)
    gecmis = gecmis + [{"role": "user", "content": kod}, {"role": "assistant", "content": cevap}]
    return gecmis, gecmis, metrik_html_olustur()


def takip_sorusu(mesaj, gecmis):
    if not mesaj.strip():
        return gecmis, gecmis, ""
    try:
        if any(k in mesaj.lower() for k in ["duzelt", "fix", "repair", "onar", "correct"]):
            son_kod = ""
            for msg in reversed(gecmis):
                if msg["role"] == "user" and len(msg["content"]) > 20:
                    son_kod = msg["content"]
                    break
            if son_kod:
                dil = dil_tespit(son_kod)
                duzeltilmis, aciklamalar = code_repair(son_kod, dil)
                cevap = (
                    "**Suggested Fix**" + chr(10) + chr(10)
                    + "```" + dil + chr(10) + duzeltilmis + chr(10) + "```" + chr(10) + chr(10)
                    + "**Yapilan degisiklikler:**" + chr(10)
                    + chr(10).join(["- " + a for a in aciklamalar])
                )
            else:
                cevap = "Duzeltilecek kod bulunamadi. Once sol tarafa kodu yapistirin."
        else:
            sonuc = code_review(mesaj, dil_tespit(mesaj))
            cevap = sonuc["review"]
    except Exception as e:
        cevap = "Hata: " + str(e)
    gecmis = gecmis + [{"role": "user", "content": mesaj}, {"role": "assistant", "content": cevap}]
    return gecmis, gecmis, ""


def temizle(gecmis):
    return [], [], metrik_html_olustur()


with gr.Blocks(css=CSS, title="AI Code Reviewer") as demo:
    state = gr.State([])

    gr.HTML("""
    <div id='header' style='background:#0f0f1a;border-bottom:1px solid #1e1e2e;padding:12px 24px;
         display:flex;align-items:center;justify-content:space-between;width:100%;position:sticky;top:0;z-index:100;'>
        <div style='display:flex;align-items:center;gap:12px;'>
            <div style='background:linear-gradient(135deg,#7C3AED,#4F46E5);width:32px;height:32px;
                 border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:16px;'>✦</div>
            <span id='header-title' style='color:#e2e8f0;font-size:18px;font-weight:700;'>AI Code Reviewer</span>
            <span style='color:#475569;font-size:11px;padding:3px 8px;border:1px solid #1e1e2e;border-radius:4px;'>GraphBERT</span>
            <span style='color:#475569;font-size:11px;padding:3px 8px;border:1px solid #1e1e2e;border-radius:4px;'>CodeT5+</span>
            <span style='color:#475569;font-size:11px;padding:3px 8px;border:1px solid #1e1e2e;border-radius:4px;'>RAG</span>
        </div>
        <button onclick='toggleTheme()' id='theme-btn'
            style='background:#1a1a2e;border:1px solid #2d2d3a;border-radius:8px;
                   padding:6px 14px;color:#94a3b8;font-size:13px;cursor:pointer;'>🌙 Dark</button>
    </div>
    <script>
    var isLight = false;
    function toggleTheme() {
        isLight = !isLight;
        var btn=document.getElementById('theme-btn'),
            header=document.getElementById('header'),
            title=document.getElementById('header-title');
        document.body.classList.toggle('light-mode', isLight);
        if (isLight) {
            header.style.background='#ffffff'; header.style.borderBottomColor='#e2e8f0';
            if(title) title.style.color='#0f172a';
            btn.innerHTML='☀️ Light'; btn.style.background='#f1f5f9'; btn.style.color='#475569';
        } else {
            header.style.background='#0f0f1a'; header.style.borderBottomColor='#1e1e2e';
            if(title) title.style.color='#e2e8f0';
            btn.innerHTML='🌙 Dark'; btn.style.background='#1a1a2e'; btn.style.color='#94a3b8';
        }
    }
    </script>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Row(equal_height=True):
                with gr.Column(scale=1, min_width=0):
                    gr.HTML("<div style='background:#0f0f1a;border:1px solid #1e1e2e;border-radius:14px 14px 0 0;padding:12px 16px;'>"
                            "<span style='color:#94a3b8;font-size:13px;font-weight:600;'>🧠 Code Input</span></div>")
                    kod_girisi = gr.Textbox(label="", lines=20,
                                            placeholder="# Kodunuzu buraya yapistirin...",
                                            show_label=False)
                    with gr.Row(elem_classes="analiz-btn"):
                        analiz_btn = gr.Button("🚀 Analyze Code")

                with gr.Column(scale=1, min_width=0):
                    gr.HTML("<div style='background:#0f0f1a;border:1px solid #1e1e2e;border-radius:14px 14px 0 0;"
                            "padding:12px 16px;display:flex;align-items:center;justify-content:space-between;'>"
                            "<span style='color:#94a3b8;font-size:13px;font-weight:600;'>💬 AI Code Review Assistant</span>"
                            "<span onclick='clearChat()' style='color:#475569;font-size:12px;cursor:pointer;"
                            "padding:4px 10px;border:1px solid #2d2d3a;border-radius:6px;'>🗑️ Clear Chat</span></div>")
                    chatbot = gr.Chatbot(label="", height=430, bubble_full_width=False,
                                         show_label=False, type="messages", elem_classes="chatbot-alan")
                    with gr.Row():
                        mesaj_kutusu = gr.Textbox(
                            placeholder="'neden?', 'nasil duzeltiririm?' gibi sorular yazin",
                            label="", scale=5, container=False)
                        with gr.Column(scale=1, min_width=80, elem_classes="send-btn"):
                            sor_btn = gr.Button("Gonder")
                    temizle_btn = gr.Button("temizle_gizli", visible=False, elem_id="temizle-btn")

            gr.HTML("<script>function clearChat(){var b=document.getElementById('temizle-btn');if(b)b.click();}</script>")
            metrik_goster = gr.HTML(metrik_html_olustur())
            gr.HTML("<div style='text-align:center;color:#334155;font-size:11px;padding:12px 0;'>"
                    "© 2024 AI Code Reviewer • NLP Project • YZM358 💜</div>")

    analiz_btn.click(analiz_et, [kod_girisi, state], [chatbot, state, metrik_goster])
    sor_btn.click(takip_sorusu, [mesaj_kutusu, state], [chatbot, state, mesaj_kutusu])
    mesaj_kutusu.submit(takip_sorusu, [mesaj_kutusu, state], [chatbot, state, mesaj_kutusu])
    temizle_btn.click(temizle, [state], [chatbot, state, metrik_goster])

demo.launch(share=True)


/tmp/ipykernel_9842/4229884369.py:107: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="AI Code Reviewer") as demo:
/tmp/ipykernel_9842/4229884369.py:164: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(label="", height=430, bubble_full_width=False,
/tmp/ipykernel_9842/4229884369.py:164: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="", height=430, bubble_full_width=False,


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://912c3ad6312eb6aec8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
